# 3-Month Follow-Up Adherence & Resurgery Rate Trend Analysis

This notebook tracks **1-Month Follow-Up Adherence (11–45 Days)** and **1-Month Resurgery Rates (11–45 Days)** across **3 consecutive monthly datasets**.

### Clinical Definitions & Scope:
- **1-Month Follow-Up Adherence (11–45 Days)**: Evaluated strictly for visits occurring between **11 and 45 days post-surgery** (`11 <= days_after_surgery <= 45`). A primary surgery is Adherent (`YES`) if at least one follow-up visit occurred in this window.
- **1-Month Graft-Specific Resurgeries (11–45 Days)**: Evaluated strictly for re-operations occurring between **11 and 45 days post-surgery** (`11 <= days_after_surgery <= 45`), categorized into **Rebubbling / Descematopexy**, **Wound Resuturing**, and **Repeat Keratoplasty (KP)**.
- **Supportive & In-Clinic Procedures (11–45 Days)**: Reported separately and transparently (including **IOAB / Antibiotic Injections**, **Examination Under Anesthesia (EUA)**, **Tarsorrhaphy**, and **AC Wash/Reformation**) without distorting the graft revision rate.
- **Trend Comparisons**: Side-by-side comparison across Month 1, Month 2, Month 3, 3-Month Pooled Total, and Trend Change (M3 vs M1).
- **Deliverables**: Formatted Excel Workbook (`Multi_Month_Trend_Results.xlsx`) and Executive Widescreen PowerPoint Presentation (`Multi_Month_Trend_Report.pptx`).

In [ ]:
import os
import re
import io
from datetime import datetime
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pptx
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ── Specify 3 Consecutive Monthly Files ──
FILE_MONTH_1 = r'data/Given Corn Tran Surg Px  Adv and FUP Data Details-Mar2026.xlsb'
FILE_MONTH_2 = r'data/Given Corn Tran Surg Px  Adv and FUP Data Details-April2026.xlsb'
FILE_MONTH_3 = r'data/Given Corn Tran Surg Px  Adv and FUP Data Details _may.xlsb'

OUTPUT_EXCEL = r'Multi_Month_Trend_Results.xlsx'
OUTPUT_PPTX  = r'Multi_Month_Trend_Report.pptx'

print('[OK] Configuration ready.')


In [ ]:
# ── Clinical Procedure & Prefix Constants ──
CAMPUS_SHEETS = [
    'KAR Campus Data',
    'KVC Campus Data',
    'GMR Campus Data',
    'MTC Campus Data',
    'YSR Campus Data',
]

REPEAT_KP_PROCS = {
    'THERAPUETIC PENETRATING KERATOPLASTY (TH PK)',
    'INTRAOCULAR ANTIBIOTIC INJECTION (IOAB),THERAPUETIC PENETRATING KERATOPLASTY (TH PK)',
    'DESCEMETS STRIPPING AUTOMATED ENDOTHELIAL KERATOPLASTY (DSAEK)',
    'ANTERIOR VITRECTOMY,THERAPUETIC PENETRATING KERATOPLASTY (TH PK)',
    'PENETRATING KERATOPLASTY (PK)',
    'DESCEMET MEMBRANE ENDOTHLIAL KERATOPLASTY (DMEK)',
    'ECCE + IOL,PENETRATING KERATOPLASTY (PK),TARSORRAPHY',
    'DEEP ANTERIOR LAMELLAR KERATOPLASTY (DALK)',
}

PREFIX_MAP = {"PN": "P", "NP": "N", "NPC": "CC", "PNC": "CC", "NPN": "N", "PNP": "P"}

def extract_month_label(filename_or_str: str, fallback_idx: int = 1) -> str:
    """Extract month name and year (if present) from a filename string."""
    if not filename_or_str:
        return f"Month {fallback_idx}"
    months = {
        'jan': 'January', 'feb': 'February', 'mar': 'March', 'apr': 'April',
        'may': 'May', 'jun': 'June', 'jul': 'July', 'aug': 'August',
        'sep': 'September', 'oct': 'October', 'nov': 'November', 'dec': 'December'
    }
    s = str(filename_or_str).lower()
    for m_short, m_full in months.items():
        match = re.search(rf'({m_short}[a-z]*)\s*[-_]?\s*(\d{{4}})?', s)
        if match:
            year = match.group(2)
            return f"{m_full} {year}" if year else m_full
    return f"Month {fallback_idx}"


def correct_mrno(mrno) -> str:
    if pd.isna(mrno):
        return ""
    mrno_str = str(mrno).strip()
    match = re.search(r"(?:.*-)?([A-Z]+)(\d+)$", mrno_str)
    if match:
        prefix, digits = match.groups()
        new_prefix = PREFIX_MAP.get(prefix, prefix)
        return re.sub(r"([A-Z]+)(\d+)$", new_prefix + digits, mrno_str)
    return mrno_str


def categorize_repeat_surgery(proc_name, adv_proc_name=None) -> str:
    """
    Categorizes post-op re-interventions into repeat surgery categories matching
    morbidity_analysis_clean:
      - REBUBBLING
      - WOUND_RESUTURING
      - KP (Repeat Keratoplasty procedures in REPEAT_KP_PROCS)
      - Others (Minor procedures / supportive interventions)

    Apart from Others, REBUBBLING, WOUND_RESUTURING, and KP are the real repeat surgeries.
    """
    name = proc_name if (pd.notna(proc_name) and str(proc_name).strip() != "") else adv_proc_name
    if pd.isna(name):
        return "Others"
    s = str(name).strip()
    if s == "REBUBBLING":
        return "REBUBBLING"
    if s in ("WOUND RESUTURING", "WOUND_RESUTURING"):
        return "WOUND_RESUTURING"
    if s in REPEAT_KP_PROCS:
        return "KP"
    return "Others"


def load_and_clean_month_df(file_source) -> pd.DataFrame:
    """Load campus sheets from an excel file buffer or path, standardizing dates and columns."""
    dfs = []
    fname = getattr(file_source, 'name', str(file_source)).lower()
    xl_engine = 'pyxlsb' if fname.endswith('.xlsb') else None
    try:
        if isinstance(file_source, (str, os.PathLike)):
            xl = pd.ExcelFile(file_source, engine=xl_engine)
        elif hasattr(file_source, 'getvalue'):
            xl = pd.ExcelFile(io.BytesIO(file_source.getvalue()), engine=xl_engine)
        elif hasattr(file_source, 'read'):
            b = file_source.read()
            if hasattr(file_source, 'seek'):
                file_source.seek(0)
            xl = pd.ExcelFile(io.BytesIO(b), engine=xl_engine)
        else:
            xl = pd.ExcelFile(file_source, engine=xl_engine)

        available = xl.sheet_names
        matching_sheets = [s for s in available if s in CAMPUS_SHEETS or 'campus' in s.lower()]
        if not matching_sheets:
            matching_sheets = available

        for s in matching_sheets:
            df_top = pd.read_excel(xl, sheet_name=s, header=None, nrows=15)
            hdr_row = 4
            for idx, r in df_top.iterrows():
                r_vals = [str(x).lower() for x in r.values]
                if any(k in r_vals for k in ['sap_code', 'tp_mrno', 'surg_date', 'graft_health_text']):
                    hdr_row = idx
                    break
            d = pd.read_excel(xl, sheet_name=s, header=hdr_row)
            dfs.append(d)
    except Exception:
        df = pd.read_excel(file_source, engine=xl_engine)
        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)

    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)

    for col in ["visit_date", "surg_date", "dob", "fup_surg_date"]:
        if col in df.columns:
            parsed = pd.to_datetime(df[col], errors='coerce')
            numeric = pd.to_numeric(df[col], errors='coerce')
            valid_numeric = numeric.where(numeric.between(1, 100000))
            excel_dates = pd.Timestamp("1899-12-30") + pd.to_timedelta(valid_numeric, unit="D", errors='coerce')
            df[col] = parsed.fillna(excel_dates)

    if "tp_mrno" in df.columns:
        df["tp_mrno_corrected"] = df["tp_mrno"].apply(correct_mrno)
    else:
        df["tp_mrno_corrected"] = ""

    for required_col in ["surg_eye", "surg_proc_group"]:
        if required_col not in df.columns:
            df[required_col] = "UNKNOWN"

    df["id"] = (
        df["tp_mrno_corrected"].astype(str) + "_" +
        df["surg_eye"].astype(str) + "_" +
        df["surg_proc_group"].astype(str) + "_" +
        df["surg_date"].dt.strftime("%Y-%m-%d").fillna("")
    )

    df = df.dropna(how="all").dropna(subset=["id"]).sort_values(["id", "surg_date", "visit_date"])
    df["days_after_surgery"] = (df["visit_date"] - df["surg_date"]).dt.days
    return df


def process_single_month(df: pd.DataFrame) -> dict:
    """Extract primary surgeries, 1M adherence, and 1M repeat surgeries for a single month."""
    for col_req in ['advise_surg', 'fup_surg_done', 'fup_done_surg_proc', 'adv_surg_proc']:
        if col_req not in df.columns:
            df[col_req] = None

    # Identify repeat keratoplasties
    repeat_mask = (
        (df['visit_date'] > df['surg_date']) &
        (df['advise_surg'] == 'YES') &
        (df['fup_surg_done'] == 'YES') &
        (df['fup_done_surg_proc'].isin(REPEAT_KP_PROCS))
    )
    repeat_dates = (
        df.loc[repeat_mask]
        .groupby('id', as_index=False)['fup_surg_date'].min()
        .rename(columns={'fup_surg_date': 'first_repeat_keratoplasty_date'})
    )
    visit_df = df.merge(repeat_dates, on='id', how='left')
    visit_df_primary = visit_df[
        visit_df['first_repeat_keratoplasty_date'].isna() |
        (visit_df['visit_date'] < visit_df['first_repeat_keratoplasty_date'])
    ].copy()

    # One row per primary surgery
    surgery_df = visit_df_primary.groupby('id', as_index=False).first()

    # 1-Month Follow-Up Adherence (Strictly 11 <= days_after_surgery <= 45)
    fup_1m = visit_df_primary[visit_df_primary['days_after_surgery'].between(11, 45)]
    adherent_ids = set(fup_1m['id'].unique())
    surgery_df['is_adherent_1m'] = surgery_df['id'].isin(adherent_ids)

    # 1-Month Resurgeries & Re-interventions (Strictly 11 <= days_after_surgery <= 45)
    repeat_1m = visit_df_primary[
        (visit_df_primary['days_after_surgery'].between(11, 45)) &
        (visit_df_primary['advise_surg'] == 'YES') &
        (visit_df_primary['fup_surg_done'] == 'YES')
    ].copy()

    # Categorize procedures matching morbidity analysis clean logic
    repeat_1m['Repeat_Category'] = [
        categorize_repeat_surgery(row.get('fup_done_surg_proc'), row.get('adv_surg_proc'))
        for _, row in repeat_1m.iterrows()
    ]
    # Real repeat surgeries are REBUBBLING, WOUND_RESUTURING, KP (apart from Others)
    repeat_1m['is_real_repeat'] = repeat_1m['Repeat_Category'].isin(['REBUBBLING', 'WOUND_RESUTURING', 'KP'])
    repeat_1m['is_graft_resurgery'] = repeat_1m['is_real_repeat']

    # Tag primary surgeries that had real repeat surgeries vs any re-intervention (including Others)
    real_resurg_ids = set(repeat_1m[repeat_1m['is_real_repeat']]['id'].unique())
    any_reintervention_ids = set(repeat_1m['id'].unique())
    surgery_df['has_graft_resurgery_1m'] = surgery_df['id'].isin(real_resurg_ids)
    surgery_df['has_repeat_surgery_1m'] = surgery_df['id'].isin(real_resurg_ids)
    surgery_df['has_any_reintervention_1m'] = surgery_df['id'].isin(any_reintervention_ids)

    return {
        "surgery_df": surgery_df,
        "visit_df_primary": visit_df_primary,
        "repeat_1m": repeat_1m,
    }


def build_3month_trend_tables(month_results: dict[str, dict]) -> dict[str, pd.DataFrame]:
    """
    Given dict of {month_label: process_single_month_output},
    build side-by-side trend tables with explicit '1M (11–45 Days)' labeling
    and clear clinical separation of Graft Resurgeries vs Supportive Interventions.
    """
    month_keys = list(month_results.keys())
    m1, m2, m3 = month_keys[0], month_keys[1], month_keys[2]

    # Combine primary surgeries
    all_surgeries = []
    all_repeats = []
    for m in month_keys:
        sdf = month_results[m]["surgery_df"].copy()
        sdf["Month"] = m
        all_surgeries.append(sdf)

        rdf = month_results[m]["repeat_1m"].copy()
        rdf["Month"] = m
        all_repeats.append(rdf)

    comb_surg = pd.concat(all_surgeries, ignore_index=True)
    comb_repeat = pd.concat(all_repeats, ignore_index=True) if all_repeats else pd.DataFrame()

    # ─────────────────────────────────────────────────────────────
    # Table 1: Adherence Trend — Overall (11–45 Days)
    # ─────────────────────────────────────────────────────────────
    t1_rows = []
    tot_surg = {m: len(month_results[m]["surgery_df"]) for m in month_keys}
    adh_cnt = {m: month_results[m]["surgery_df"]["is_adherent_1m"].sum() for m in month_keys}
    adh_pct = {m: (adh_cnt[m] / tot_surg[m] * 100).round(2) if tot_surg[m] > 0 else 0.0 for m in month_keys}

    pooled_surg = sum(tot_surg.values())
    pooled_adh = sum(adh_cnt.values())
    pooled_pct = round((pooled_adh / pooled_surg * 100), 2) if pooled_surg > 0 else 0.0
    delta_adh = round(adh_pct[m3] - adh_pct[m1], 2)

    t1_rows.append({"Metric": "Total Primary Surgeries", m1: tot_surg[m1], m2: tot_surg[m2], m3: tot_surg[m3], "3-Month Total": pooled_surg, "Trend (M3 - M1)": tot_surg[m3] - tot_surg[m1]})
    t1_rows.append({"Metric": "1M Adherent Surgeries (11–45 Days)", m1: adh_cnt[m1], m2: adh_cnt[m2], m3: adh_cnt[m3], "3-Month Total": pooled_adh, "Trend (M3 - M1)": adh_cnt[m3] - adh_cnt[m1]})
    t1_rows.append({"Metric": "1M Adherence Rate (11–45 Days) [%]", m1: f"{adh_pct[m1]}%", m2: f"{adh_pct[m2]}%", m3: f"{adh_pct[m3]}%", "3-Month Total": f"{pooled_pct}%", "Trend (M3 - M1)": f"{delta_adh:+.2f}%"})
    df_adh_overall = pd.DataFrame(t1_rows)

    # ─────────────────────────────────────────────────────────────
    # Helper for Grouped Adherence Trend (11–45 Days)
    # ─────────────────────────────────────────────────────────────
    def _grouped_adherence_trend(group_col: str, group_label: str) -> pd.DataFrame:
        groups = sorted(comb_surg[group_col].dropna().unique())
        rows = []
        for g in groups:
            r = {group_label: g}
            g_pooled_surg = 0
            g_pooled_adh = 0
            for m in month_keys:
                msurg = month_results[m]["surgery_df"]
                sub = msurg[msurg[group_col] == g]
                cnt = len(sub)
                adh = sub["is_adherent_1m"].sum() if cnt > 0 else 0
                pct = round(adh / cnt * 100, 2) if cnt > 0 else 0.0
                r[f"{m} Surgeries"] = cnt
                r[f"{m} Adherent % (11–45d)"] = pct
                g_pooled_surg += cnt
                g_pooled_adh += adh

            pool_pct = round(g_pooled_adh / g_pooled_surg * 100, 2) if g_pooled_surg > 0 else 0.0
            r["3-Month Surgeries"] = g_pooled_surg
            r["3-Month Adherent % (11–45d)"] = pool_pct
            r["Trend (M3 vs M1)"] = round(r[f"{m3} Adherent % (11–45d)"] - r[f"{m1} Adherent % (11–45d)"], 2)
            rows.append(r)
        return pd.DataFrame(rows)

    df_adh_campus = _grouped_adherence_trend("sap_code", "Campus")
    df_adh_surg = _grouped_adherence_trend("surg_proc_group", "Surgery Procedure")

    # ─────────────────────────────────────────────────────────────
    # Table 4: Resurgery Trend — Overall & Detailed Categories (11–45 Days)
    # ─────────────────────────────────────────────────────────────
    t4_rows = []

    # Section 1: Denominator
    t4_rows.append({
        "Procedure Category": "Total Primary Surgeries",
        m1: tot_surg[m1], m2: tot_surg[m2], m3: tot_surg[m3],
        "3-Month Total": pooled_surg,
        "Trend (M3 vs M1)": tot_surg[m3] - tot_surg[m1]
    })

    # Section 2: Real Repeat Surgeries (REBUBBLING, WOUND_RESUTURING, KP)
    real_cats = ["REBUBBLING", "WOUND_RESUTURING", "KP"]
    
    # Total Real Repeat Surgeries
    g_resurg_cnt = {m: month_results[m]["surgery_df"]["has_graft_resurgery_1m"].sum() for m in month_keys}
    g_resurg_pct = {m: round(g_resurg_cnt[m] / tot_surg[m] * 100, 2) if tot_surg[m] > 0 else 0.0 for m in month_keys}
    p_g_cnt = sum(g_resurg_cnt.values())
    p_g_pct = round(p_g_cnt / pooled_surg * 100, 2) if pooled_surg > 0 else 0.0

    t4_rows.append({
        "Procedure Category": "Total Repeat Surgeries (Real: Rebubbling, Resuturing, KP) [11–45 Days]",
        m1: f"{g_resurg_cnt[m1]} ({g_resurg_pct[m1]}%)",
        m2: f"{g_resurg_cnt[m2]} ({g_resurg_pct[m2]}%)",
        m3: f"{g_resurg_cnt[m3]} ({g_resurg_pct[m3]}%)",
        "3-Month Total": f"{p_g_cnt} ({p_g_pct}%)",
        "Trend (M3 vs M1)": f"{g_resurg_pct[m3] - g_resurg_pct[m1]:+.2f}%"
    })

    for cat in real_cats:
        r = {"Procedure Category": f"  • {cat}"}
        p_cnt = 0
        for m in month_keys:
            reps = month_results[m]["repeat_1m"]
            cnt = (reps["Repeat_Category"] == cat).sum()
            pct = round(cnt / tot_surg[m] * 100, 2) if tot_surg[m] > 0 else 0.0
            r[m] = f"{cnt} ({pct}%)"
            p_cnt += cnt
        r["3-Month Total"] = f"{p_cnt} ({round(p_cnt / pooled_surg * 100, 2)}%)"
        m1_cnt = (month_results[m1]["repeat_1m"]["Repeat_Category"] == cat).sum()
        m3_cnt = (month_results[m3]["repeat_1m"]["Repeat_Category"] == cat).sum()
        r["Trend (M3 vs M1)"] = f"{m3_cnt - m1_cnt:+d}"
        t4_rows.append(r)

    # Section 3: Others (Minor Procedures)
    r_oth = {"Procedure Category": "Others (Minor Procedures)"}
    p_oth_cnt = 0
    for m in month_keys:
        reps = month_results[m]["repeat_1m"]
        cnt = (reps["Repeat_Category"] == "Others").sum()
        pct = round(cnt / tot_surg[m] * 100, 2) if tot_surg[m] > 0 else 0.0
        r_oth[m] = f"{cnt} ({pct}%)"
        p_oth_cnt += cnt
    r_oth["3-Month Total"] = f"{p_oth_cnt} ({round(p_oth_cnt / pooled_surg * 100, 2)}%)"
    m1_oth_cnt = (month_results[m1]["repeat_1m"]["Repeat_Category"] == "Others").sum()
    m3_oth_cnt = (month_results[m3]["repeat_1m"]["Repeat_Category"] == "Others").sum()
    r_oth["Trend (M3 vs M1)"] = f"{m3_oth_cnt - m1_oth_cnt:+d}"
    t4_rows.append(r_oth)

    # Section 4: All Re-interventions Combined (Real + Others)
    all_re_cnt = {m: len(month_results[m]["repeat_1m"]) for m in month_keys}
    p_all_re = sum(all_re_cnt.values())
    t4_rows.append({
        "Procedure Category": "Total Combined Re-interventions (All Procedures)",
        m1: f"{all_re_cnt[m1]} ({round(all_re_cnt[m1]/tot_surg[m1]*100, 2)}%)",
        m2: f"{all_re_cnt[m2]} ({round(all_re_cnt[m2]/tot_surg[m2]*100, 2)}%)",
        m3: f"{all_re_cnt[m3]} ({round(all_re_cnt[m3]/tot_surg[m3]*100, 2)}%)",
        "3-Month Total": f"{p_all_re} ({round(p_all_re/pooled_surg*100, 2)}%)",
        "Trend (M3 vs M1)": f"{all_re_cnt[m3] - all_re_cnt[m1]:+d}"
    })

    df_rep_overall = pd.DataFrame(t4_rows)

    # ─────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────
    # Helper for Grouped Resurgery Breakdown (Overall & Categories) (11–45 Days)
    # ─────────────────────────────────────────────────────────────
    def _grouped_resurgery_breakdown(group_col: str, group_label: str) -> pd.DataFrame:
        groups = sorted(list(set().union(*[month_results[m]["surgery_df"][group_col].dropna().unique() for m in month_keys])))
        rows = []
        for g in groups:
            tot_s = {m: len(month_results[m]["surgery_df"][month_results[m]["surgery_df"][group_col] == g]) for m in month_keys}
            pool_s = sum(tot_s.values())
            if pool_s == 0:
                continue

            # 1. Total Primary Surgeries
            rows.append({
                group_label: g,
                "Category": "Total Primary Surgeries",
                m1: tot_s[m1], m2: tot_s[m2], m3: tot_s[m3],
                "3-Month Total": pool_s,
                "Trend (M3 vs M1)": tot_s[m3] - tot_s[m1]
            })

            # 2. Total Real Repeat Surgeries
            real_cnt = {m: month_results[m]["surgery_df"][(month_results[m]["surgery_df"][group_col] == g) & (month_results[m]["surgery_df"]["has_graft_resurgery_1m"])].shape[0] for m in month_keys}
            pool_real = sum(real_cnt.values())
            pcts = {m: round(real_cnt[m] / tot_s[m] * 100, 2) if tot_s[m] > 0 else 0.0 for m in month_keys}
            p_pct = round(pool_real / pool_s * 100, 2) if pool_s > 0 else 0.0
            rows.append({
                group_label: g,
                "Category": "  Total Real Repeat Surgeries (Rebubbling, Resuturing, KP)",
                m1: f"{real_cnt[m1]} ({pcts[m1]}%)",
                m2: f"{real_cnt[m2]} ({pcts[m2]}%)",
                m3: f"{real_cnt[m3]} ({pcts[m3]}%)",
                "3-Month Total": f"{pool_real} ({p_pct}%)",
                "Trend (M3 vs M1)": f"{pcts[m3] - pcts[m1]:+.2f}%"
            })

            # 3. Specific Real Repeat Categories: REBUBBLING, WOUND_RESUTURING, KP
            for sub in ["REBUBBLING", "WOUND_RESUTURING", "KP"]:
                cnts = {m: month_results[m]["repeat_1m"][(month_results[m]["repeat_1m"][group_col] == g) & (month_results[m]["repeat_1m"]["Repeat_Category"] == sub)].shape[0] for m in month_keys}
                p_cnt = sum(cnts.values())
                rows.append({
                    group_label: g,
                    "Category": f"    • {sub}",
                    m1: cnts[m1], m2: cnts[m2], m3: cnts[m3],
                    "3-Month Total": p_cnt,
                    "Trend (M3 vs M1)": cnts[m3] - cnts[m1]
                })

            # 4. Others (Minor Procedures)
            oth_cnt = {m: month_results[m]["repeat_1m"][(month_results[m]["repeat_1m"][group_col] == g) & (month_results[m]["repeat_1m"]["Repeat_Category"] == "Others")].shape[0] for m in month_keys}
            p_oth = sum(oth_cnt.values())
            oth_pcts = {m: round(oth_cnt[m] / tot_s[m] * 100, 2) if tot_s[m] > 0 else 0.0 for m in month_keys}
            p_oth_pct = round(p_oth / pool_s * 100, 2) if pool_s > 0 else 0.0
            rows.append({
                group_label: g,
                "Category": "  Others (Minor Procedures)",
                m1: f"{oth_cnt[m1]} ({oth_pcts[m1]}%)",
                m2: f"{oth_cnt[m2]} ({oth_pcts[m2]}%)",
                m3: f"{oth_cnt[m3]} ({oth_pcts[m3]}%)",
                "3-Month Total": f"{p_oth} ({p_oth_pct}%)",
                "Trend (M3 vs M1)": f"{oth_pcts[m3] - oth_pcts[m1]:+.2f}%" if tot_s[m3] > 0 and tot_s[m1] > 0 else "---"
            })

            # 5. Total Combined Re-interventions
            all_re = {m: month_results[m]["repeat_1m"][month_results[m]["repeat_1m"][group_col] == g].shape[0] for m in month_keys}
            p_all = sum(all_re.values())
            all_pcts = {m: round(all_re[m] / tot_s[m] * 100, 2) if tot_s[m] > 0 else 0.0 for m in month_keys}
            p_all_pct = round(p_all / pool_s * 100, 2) if pool_s > 0 else 0.0
            rows.append({
                group_label: g,
                "Category": "  Total Combined Re-interventions (All)",
                m1: f"{all_re[m1]} ({all_pcts[m1]}%)",
                m2: f"{all_re[m2]} ({all_pcts[m2]}%)",
                m3: f"{all_re[m3]} ({all_pcts[m3]}%)",
                "3-Month Total": f"{p_all} ({p_all_pct}%)",
                "Trend (M3 vs M1)": f"{all_re[m3] - all_re[m1]:+d}"
            })

        return pd.DataFrame(rows)

    df_rep_campus = _grouped_resurgery_breakdown("sap_code", "Campus")
    df_rep_surg = _grouped_resurgery_breakdown("surg_proc_group", "Surgery Procedure")

    return {
        "1_Trend_Adherence_Overall": df_adh_overall,
        "2_Trend_Adherence_Campus": df_adh_campus,
        "3_Trend_Adherence_SurgType": df_adh_surg,
        "4_Trend_Resurgery_Overall": df_rep_overall,
        "5_Trend_Resurgery_Campus": df_rep_campus,
        "6_Trend_Resurgery_SurgType": df_rep_surg,
    }


def create_trend_excel(trend_tables: dict[str, pd.DataFrame]) -> io.BytesIO:
    """Creates an executive formatted openpyxl workbook with all 6 trend sheets."""
    output = io.BytesIO()
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    SUBHEADER_FILL = PatternFill(start_color="2F5597", end_color="2F5597", fill_type="solid")
    HEADER_FONT = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    DATA_FONT = Font(name="Segoe UI", size=10, bold=False, color="000000")
    BOLD_FONT = Font(name="Segoe UI", size=10, bold=True, color="000000")
    TOTAL_ROW_FILL = PatternFill(start_color="EBF1F5", end_color="EBF1F5", fill_type="solid")
    ALT_ROW_FILL = PatternFill(start_color="F8F9FA", end_color="F8F9FA", fill_type="solid")
    WHITE_ROW_FILL = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")

    THIN_SIDE = Side(border_style="thin", color="D9D9D9")
    MEDIUM_BOTTOM = Side(border_style="medium", color="1F4E78")
    DOUBLE_BOTTOM = Side(border_style="double", color="1F4E78")

    DATA_BORDER = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=THIN_SIDE)
    HEADER_BORDER = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=MEDIUM_BOTTOM)
    TOTAL_BORDER = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=DOUBLE_BOTTOM)

    TAB_COLORS = {
        "1": "2B579A", "2": "27AE60", "3": "D35400",
        "4": "C0392B", "5": "8E44AD", "6": "2980B9"
    }

    for sheet_name, df in trend_tables.items():
        ws = wb.create_sheet(title=sheet_name[:31])
        prefix = sheet_name.split("_")[0]
        if prefix in TAB_COLORS:
            ws.sheet_properties.tabColor = TAB_COLORS[prefix]
        ws.views.sheetView[0].showGridLines = True
        ws.freeze_panes = 'A3'

        # Row 1: Note Banner explaining 1M window
        headers = list(df.columns)
        n_cols = len(headers)
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
        note_cell = ws.cell(row=1, column=1, value="Note: 1-Month Follow-Up and Resurgery metrics are evaluated strictly between 11 and 45 days post-surgery.")
        note_cell.font = Font(name="Segoe UI", size=9, italic=True, color="555555")
        note_cell.alignment = Alignment(horizontal="left", vertical="center")
        ws.row_dimensions[1].height = 18

        # Row 2: Headers
        for col_idx, h in enumerate(headers, 1):
            cell = ws.cell(row=2, column=col_idx, value=h)
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = HEADER_BORDER
        ws.row_dimensions[2].height = 28

        # Data rows starting at row 3
        for row_idx, row_data in enumerate(df.itertuples(index=False), 3):
            ws.row_dimensions[row_idx].height = 20
            first_val = str(row_data[0] or "").lower()
            is_total = "total" in first_val or "overall" in first_val or "combined" in first_val
            is_section_header = "---" in str(row_data[1] or "")
            
            if is_section_header:
                r_fill = SUBHEADER_FILL
                r_font = Font(name="Segoe UI", size=10, bold=True, color="FFFFFF")
            elif is_total:
                r_fill = TOTAL_ROW_FILL
                r_font = BOLD_FONT
            elif row_idx % 2 == 0:
                r_fill = ALT_ROW_FILL
                r_font = DATA_FONT
            else:
                r_fill = WHITE_ROW_FILL
                r_font = DATA_FONT

            r_border = TOTAL_BORDER if is_total else DATA_BORDER

            for col_idx, val in enumerate(row_data, 1):
                cell = ws.cell(row=row_idx, column=col_idx, value=val)
                cell.fill = r_fill
                cell.font = r_font
                cell.border = r_border
                h_name = headers[col_idx - 1]

                if "%" in h_name or "rate" in h_name.lower():
                    cell.alignment = Alignment(horizontal="right", vertical="center")
                    if isinstance(val, (int, float)):
                        cell.number_format = '0.00"%"'
                elif any(k in h_name for k in ["Count", "Surgeries", "Total"]) and "rate" not in h_name.lower():
                    cell.alignment = Alignment(horizontal="right", vertical="center")
                    if isinstance(val, (int, float)):
                        cell.number_format = '#,##0'
                elif "trend" in h_name.lower() or "δ" in h_name.lower():
                    cell.alignment = Alignment(horizontal="right", vertical="center")
                    if isinstance(val, (int, float)):
                        cell.number_format = '+0.00"%";-0.00"%";0.00"%"'
                else:
                    cell.alignment = Alignment(horizontal="left", vertical="center")

        # Column widths
        for col_idx in range(1, len(headers) + 1):
            col_letter = get_column_letter(col_idx)
            max_len = max(len(str(ws.cell(row=r, column=col_idx).value or '')) for r in range(2, ws.max_row + 1))
            h_len = len(headers[col_idx - 1])
            ws.column_dimensions[col_letter].width = min(max(max_len + 4, h_len + 4, 12), 40)

    wb.save(output)
    output.seek(0)
    return output


def _style_pptx_slide_header(slide, title_text: str, subtitle_text: str = "Evaluation Criteria: 1-Month Follow-Up & Resurgeries strictly between 11 and 45 Days Post-Surgery"):
    """Adds a standard executive header banner to a PowerPoint slide."""
    header_box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), Inches(13.333), Inches(1.1))
    header_box.fill.solid()
    header_box.fill.fore_color.rgb = RGBColor(31, 78, 120)  # Navy Blue
    header_box.line.fill.background()

    tf = header_box.text_frame
    tf.word_wrap = True
    tf.margin_left = Inches(0.5)
    tf.margin_top = Inches(0.12)

    p1 = tf.paragraphs[0]
    p1.text = title_text
    p1.font.size = Pt(20)
    p1.font.bold = True
    p1.font.color.rgb = RGBColor(255, 255, 255)

    p2 = tf.add_paragraph()
    p2.text = f"Criteria: {subtitle_text}"
    p2.font.size = Pt(10)
    p2.font.italic = True
    p2.font.color.rgb = RGBColor(210, 230, 250)


def _add_kpi_card(slide, left, top, width, height, title: str, value: str, subtext: str, border_color: RGBColor):
    """Draws an executive KPI card on a PowerPoint slide."""
    card = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, left, top, width, height)
    card.fill.solid()
    card.fill.fore_color.rgb = RGBColor(255, 255, 255)
    card.line.color.rgb = border_color
    card.line.width = Pt(2)

    tf = card.text_frame
    tf.word_wrap = True
    tf.margin_left = Inches(0.2)
    tf.margin_right = Inches(0.2)
    tf.margin_top = Inches(0.15)

    p1 = tf.paragraphs[0]
    p1.text = title.upper()
    p1.font.size = Pt(9.5)
    p1.font.bold = True
    p1.font.color.rgb = RGBColor(100, 110, 120)

    p2 = tf.add_paragraph()
    p2.text = value
    p2.font.size = Pt(22)
    p2.font.bold = True
    p2.font.color.rgb = border_color

    p3 = tf.add_paragraph()
    p3.text = subtext
    p3.font.size = Pt(8.5)
    p3.font.color.rgb = RGBColor(120, 130, 140)


def create_trend_pptx(trend_tables: dict[str, pd.DataFrame], month_names: list[str]) -> io.BytesIO:
    """
    Renders an executive, widescreen (16:9) PowerPoint presentation (PPTX)
    with high-resolution charts, KPI cards, and detailed resurgery breakdowns.
    Explicitly highlights the 1-Month (11–45 Days) evaluation criteria across all slides.
    """
    prs = pptx.Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank_layout = prs.slide_layouts[6]

    m1, m2, m3 = month_names[0], month_names[1], month_names[2]
    months = [m1, m2, m3]

    c_blue = "#1F4E78"
    c_green = "#27AE60"
    c_red = "#C0392B"
    c_purple = "#8E44AD"
    c_gray = "#7F8C8D"

    df_adh_ov = trend_tables["1_Trend_Adherence_Overall"]
    df_adh_camp = trend_tables["2_Trend_Adherence_Campus"]
    df_adh_surg = trend_tables["3_Trend_Adherence_SurgType"]
    df_rep_ov = trend_tables["4_Trend_Resurgery_Overall"]
    df_rep_camp = trend_tables["5_Trend_Resurgery_Campus"]
    df_rep_surg = trend_tables["6_Trend_Resurgery_SurgType"]

    # ─────────────────────────────────────────────────────────────
    # SLIDE 1: Executive Title & 3-Month KPI Overview
    # ─────────────────────────────────────────────────────────────
    s1 = prs.slides.add_slide(blank_layout)
    _style_pptx_slide_header(
        s1,
        "Corneal Transplantation Quality Trends | Multi-Month Executive Report",
        "Clinical Window: 1-Month Follow-Up Adherence & Resurgeries strictly evaluated between 11 and 45 Days Post-Surgery"
    )

    # 4 KPI Cards across top
    tot_surg_str = str(df_adh_ov.loc[df_adh_ov["Metric"] == "Total Primary Surgeries", "3-Month Total"].values[0])
    adh_rate_str = str(df_adh_ov.loc[df_adh_ov["Metric"] == "1M Adherence Rate (11–45 Days) [%]", "3-Month Total"].values[0])
    adh_delta_str = str(df_adh_ov.loc[df_adh_ov["Metric"] == "1M Adherence Rate (11–45 Days) [%]", "Trend (M3 - M1)"].values[0])

    graft_resurg_val = str(df_rep_ov.loc[df_rep_ov["Procedure Category"].str.startswith("Total Repeat Surgeries"), "3-Month Total"].values[0])
    comb_resurg_val = str(df_rep_ov.loc[df_rep_ov["Procedure Category"].str.startswith("Total Combined Re-interventions"), "3-Month Total"].values[0])

    card_w = Inches(2.8)
    card_h = Inches(1.3)
    top_pos = Inches(1.3)

    _add_kpi_card(s1, Inches(0.6), top_pos, card_w, card_h, "3-Month Primary Surgeries", tot_surg_str, f"Across {m1}, {m2}, {m3}", RGBColor(31, 78, 120))
    _add_kpi_card(s1, Inches(3.7), top_pos, card_w, card_h, "1M Adherence Rate (11–45d)", adh_rate_str, f"3-Month Pooled (Trend: {adh_delta_str})", RGBColor(39, 174, 96))
    _add_kpi_card(s1, Inches(6.8), top_pos, card_w, card_h, "1M Repeat Surgery Rate", graft_resurg_val, "REBUBBLING, RESUTURING, KP", RGBColor(192, 57, 43))
    _add_kpi_card(s1, Inches(9.9), top_pos, card_w, card_h, "All Re-interventions", comb_resurg_val, "Includes Minor (Others)", RGBColor(142, 68, 173))

    # Dual Chart Overview (Adherence vs Repeat Surgery Rate)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.0), dpi=180)
    adh_vals = [float(str(df_adh_ov.loc[df_adh_ov["Metric"] == "1M Adherence Rate (11–45 Days) [%]", m].values[0]).replace('%', '')) for m in months]
    ax1.plot(months, adh_vals, marker='o', linewidth=3, markersize=8, color=c_green)
    for i, v in enumerate(adh_vals):
        ax1.annotate(f"{v:.1f}%", (months[i], v), textcoords="offset points", xytext=(0, 8), ha='center', fontweight='bold', fontsize=9, color=c_green)
    ax1.set_title("1-Month Follow-Up Adherence Rate (%) [11–45 Days]", fontsize=11, fontweight='bold', color=c_blue)
    ax1.set_ylabel("Adherence Rate (%)", fontsize=9)
    ax1.set_ylim(0, 105)
    ax1.grid(True, linestyle="--", alpha=0.4)

    # Extract repeat surgery rate %
    g_resurg_rates = []
    for m in months:
        val_str = str(df_rep_ov.loc[df_rep_ov["Procedure Category"].str.startswith("Total Repeat Surgeries"), m].values[0])
        match = re.search(r'\(([\d.]+)%\)', val_str)
        g_resurg_rates.append(float(match.group(1)) if match else 0.0)

    ax2.plot(months, g_resurg_rates, marker='s', linewidth=3, markersize=8, color=c_red)
    for i, v in enumerate(g_resurg_rates):
        ax2.annotate(f"{v:.2f}%", (months[i], v), textcoords="offset points", xytext=(0, 8), ha='center', fontweight='bold', fontsize=9, color=c_red)
    ax2.set_title("1-Month Repeat Surgery Rate (%) [11–45 Days]", fontsize=11, fontweight='bold', color=c_blue)
    ax2.set_ylabel("Repeat Surgery Rate (%)", fontsize=9)
    ax2.set_ylim(0, max(g_resurg_rates + [5]) * 1.35)
    ax2.grid(True, linestyle="--", alpha=0.4)

    plt.tight_layout()
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', dpi=180)
    plt.close(fig)
    img_buf.seek(0)
    s1.shapes.add_picture(img_buf, Inches(0.6), Inches(2.8), Inches(12.1), Inches(4.3))

    # ─────────────────────────────────────────────────────────────
    # SLIDE 2: 1-Month Follow-Up Adherence (11–45 Days) — Campus Trends
    # ─────────────────────────────────────────────────────────────
    s2 = prs.slides.add_slide(blank_layout)
    _style_pptx_slide_header(s2, "1-Month Follow-Up Adherence Rate by Campus (11–45 Days Post-Surgery)")

    fig, ax = plt.subplots(figsize=(11.5, 5.2), dpi=180)
    campuses = df_adh_camp["Campus"].tolist()
    x = np.arange(len(campuses))
    width = 0.25

    colors = [c_blue, c_green, c_purple]
    for idx, m in enumerate(months):
        rates = df_adh_camp[f"{m} Adherent % (11–45d)"].tolist()
        bars = ax.bar(x + (idx - 1) * width, rates, width, label=m, color=colors[idx % len(colors)], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_title("Campus Adherence Rate Trend across 3 Months (Window: 11–45 Days)", fontsize=13, fontweight='bold', color=c_blue, pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(campuses, fontsize=10, fontweight='bold')
    ax.set_ylabel("Adherence Rate (%)", fontsize=10)
    ax.set_ylim(0, 115)
    ax.legend(frameon=True, facecolor="#F8F9FA", loc="upper right")
    ax.grid(True, axis='y', linestyle="--", alpha=0.4)

    plt.tight_layout()
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', dpi=180)
    plt.close(fig)
    img_buf.seek(0)
    s2.shapes.add_picture(img_buf, Inches(0.8), Inches(1.4), Inches(11.7), Inches(5.6))

    # ─────────────────────────────────────────────────────────────
    # SLIDE 3: 1-Month Follow-Up Adherence (11–45 Days) by Surgery Procedure
    # ─────────────────────────────────────────────────────────────
    s3 = prs.slides.add_slide(blank_layout)
    _style_pptx_slide_header(s3, "1-Month Follow-Up Adherence by Surgical Procedure (11–45 Days Post-Surgery)")

    fig, ax = plt.subplots(figsize=(11.5, 5.2), dpi=180)
    procs = df_adh_surg["Surgery Procedure"].tolist()
    x = np.arange(len(procs))
    width = 0.25

    for idx, m in enumerate(months):
        rates = df_adh_surg[f"{m} Adherent % (11–45d)"].tolist()
        bars = ax.bar(x + (idx - 1) * width, rates, width, label=m, color=colors[idx % len(colors)], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_title("Surgery Procedure Adherence Comparison (Window: 11–45 Days)", fontsize=13, fontweight='bold', color=c_blue, pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(procs, fontsize=9.5, fontweight='bold', rotation=15)
    ax.set_ylabel("Adherence Rate (%)", fontsize=10)
    ax.set_ylim(0, 115)
    ax.legend(frameon=True, facecolor="#F8F9FA", loc="upper right")
    ax.grid(True, axis='y', linestyle="--", alpha=0.4)

    plt.tight_layout()
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', dpi=180)
    plt.close(fig)
    img_buf.seek(0)
    s3.shapes.add_picture(img_buf, Inches(0.8), Inches(1.4), Inches(11.7), Inches(5.6))

    # ─────────────────────────────────────────────────────────────
    # SLIDE 4: 1-Month Graft Resurgery Rate (11–45 Days) — Campus Trends
    # ─────────────────────────────────────────────────────────────
    s4 = prs.slides.add_slide(blank_layout)
    _style_pptx_slide_header(
        s4,
        "1-Month Graft Resurgery Rate by Campus (11–45 Days Post-Surgery)",
        "Graft-Specific Resurgeries: Rebubbling / Descematopexy, Wound Resuturing, and Repeat Keratoplasty (KP)"
    )

    fig, ax = plt.subplots(figsize=(11.5, 5.2), dpi=180)
    sub_camp = df_rep_camp[df_rep_camp["Category"].str.contains("Total Real Repeat")]
    campuses_rep = sub_camp["Campus"].tolist()
    x = np.arange(len(campuses_rep))
    width = 0.25

    for idx, m in enumerate(months):
        rates = []
        for val_str in sub_camp[m]:
            match = re.search(r'\(([\d.]+)%\)', str(val_str))
            rates.append(float(match.group(1)) if match else 0.0)
        bars = ax.bar(x + (idx - 1) * width, rates, width, label=m, color=colors[idx % len(colors)], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

    ax.set_title("Real Repeat Surgery Rate by Campus across 3 Months (Window: 11–45 Days)", fontsize=13, fontweight='bold', color=c_blue, pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(campuses_rep, fontsize=10, fontweight='bold')
    ax.set_ylabel("Repeat Surgery Rate (%)", fontsize=10)
    ax.set_ylim(0, max([bar.get_height() for bar in ax.patches] + [6]) * 1.3)
    ax.legend(frameon=True, facecolor="#F8F9FA", loc="upper right")
    ax.grid(True, axis='y', linestyle="--", alpha=0.4)

    plt.tight_layout()
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', dpi=180)
    plt.close(fig)
    img_buf.seek(0)
    s4.shapes.add_picture(img_buf, Inches(0.8), Inches(1.4), Inches(11.7), Inches(5.6))

    # ─────────────────────────────────────────────────────────────
    # SLIDE 5: Resurgery Breakdown (11–45 Days)
    # ─────────────────────────────────────────────────────────────
    s5 = prs.slides.add_slide(blank_layout)
    _style_pptx_slide_header(
        s5,
        "Repeat Surgery Breakdown (11–45 Days Post-Surgery)",
        "Real Repeat Surgeries (REBUBBLING, WOUND_RESUTURING, KP) vs. Others (Minor Procedures)"
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.8), dpi=180)
    
    # Left: Real Repeat Surgeries Breakdown
    real_labels = ["REBUBBLING", "WOUND_RESUTURING", "KP"]
    x1 = np.arange(len(real_labels))
    width = 0.25

    for idx, m in enumerate(months):
        cnts = []
        for rk in real_labels:
            val_str = str(df_rep_ov.loc[df_rep_ov["Procedure Category"] == f"  • {rk}", m].values[0])
            m_cnt = int(re.match(r'(\d+)', val_str).group(1)) if re.match(r'(\d+)', val_str) else 0
            cnts.append(m_cnt)
        bars = ax1.bar(x1 + (idx - 1) * width, cnts, width, label=m, color=colors[idx % len(colors)], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax1.annotate(f"{int(h)}", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

    ax1.set_title("Real Repeat Surgeries (REBUBBLING, WOUND_RESUTURING, KP)", fontsize=11, fontweight='bold', color=c_blue)
    ax1.set_xticks(x1)
    ax1.set_xticklabels(real_labels, fontsize=9.5, fontweight='bold')
    ax1.set_ylabel("Number of Procedures", fontsize=9)
    ax1.legend(frameon=True, facecolor="#F8F9FA")
    ax1.grid(True, axis='y', linestyle="--", alpha=0.4)

    # Right: Real Repeat Surgeries vs Others (Minor Procedures)
    comp_labels = ["Real Repeat Surgeries", "Others (Minor)"]
    x2 = np.arange(len(comp_labels))

    for idx, m in enumerate(months):
        cnts = []
        val_real = str(df_rep_ov.loc[df_rep_ov["Procedure Category"].str.startswith("Total Repeat Surgeries"), m].values[0])
        m_real = int(re.match(r'(\d+)', val_real).group(1)) if re.match(r'(\d+)', val_real) else 0
        cnts.append(m_real)

        val_oth = str(df_rep_ov.loc[df_rep_ov["Procedure Category"].str.startswith("Others"), m].values[0])
        m_oth = int(re.match(r'(\d+)', val_oth).group(1)) if re.match(r'(\d+)', val_oth) else 0
        cnts.append(m_oth)

        bars = ax2.bar(x2 + (idx - 1) * width, cnts, width, label=m, color=colors[idx % len(colors)], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax2.annotate(f"{int(h)}", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

    ax2.set_title("Real Repeat Surgeries vs. Others (Minor Procedures)", fontsize=11, fontweight='bold', color=c_blue)
    ax2.set_xticks(x2)
    ax2.set_xticklabels(comp_labels, fontsize=9.5, fontweight='bold')
    ax2.set_ylabel("Number of Procedures", fontsize=9)
    ax2.legend(frameon=True, facecolor="#F8F9FA")
    ax2.grid(True, axis='y', linestyle="--", alpha=0.4)

    plt.tight_layout()
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', dpi=180)
    plt.close(fig)
    img_buf.seek(0)
    s5.shapes.add_picture(img_buf, Inches(0.8), Inches(1.4), Inches(11.7), Inches(5.6))

    # ─────────────────────────────────────────────────────────────
    # SLIDE 6: Resurgery Rate by Primary Surgery & Executive Matrix
    # ─────────────────────────────────────────────────────────────
    s6 = prs.slides.add_slide(blank_layout)
    _style_pptx_slide_header(
        s6,
        "Graft Resurgery Rate by Primary Surgical Procedure (11–45 Days Post-Surgery)",
        "Summary Scorecard & Surgical Procedure Risk Distribution"
    )

    fig, ax = plt.subplots(figsize=(11.5, 5.2), dpi=180)
    sub_surg = df_rep_surg[df_rep_surg["Category"].str.contains("Total Real Repeat")]
    procs_rep = sub_surg["Surgery Procedure"].tolist()
    x = np.arange(len(procs_rep))
    width = 0.25

    for idx, m in enumerate(months):
        rates = []
        for val_str in sub_surg[m]:
            match = re.search(r'\(([\d.]+)%\)', str(val_str))
            rates.append(float(match.group(1)) if match else 0.0)
        bars = ax.bar(x + (idx - 1) * width, rates, width, label=m, color=colors[idx % len(colors)], alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_title("Real Repeat Surgery Rate by Procedure Group across 3 Months (Window: 11–45 Days)", fontsize=13, fontweight='bold', color=c_blue, pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(procs_rep, fontsize=9.5, fontweight='bold', rotation=15)
    ax.set_ylabel("Repeat Surgery Rate (%)", fontsize=10)
    ax.set_ylim(0, max([bar.get_height() for bar in ax.patches] + [6]) * 1.3)
    ax.legend(frameon=True, facecolor="#F8F9FA", loc="upper right")
    ax.grid(True, axis='y', linestyle="--", alpha=0.4)

    plt.tight_layout()
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', dpi=180)
    plt.close(fig)
    img_buf.seek(0)
    s6.shapes.add_picture(img_buf, Inches(0.8), Inches(1.4), Inches(11.7), Inches(5.6))

    output = io.BytesIO()
    prs.save(output)
    output.seek(0)
    return output




In [ ]:
# ── Extract Month Labels ──
input_files = [FILE_MONTH_1, FILE_MONTH_2, FILE_MONTH_3]
month_labels = [extract_month_label(f, idx + 1) for idx, f in enumerate(input_files)]

print('Detected Month Labels:')
for i, (m, f) in enumerate(zip(month_labels, input_files), 1):
    print(f'  Month {i}: {m}  <--  {os.path.basename(f)}')

# ── Process Each Monthly Dataset ──
month_results = {}
for label, fpath in zip(month_labels, input_files):
    print(f'\nLoading & processing {label}...')
    df_month = load_and_clean_month_df(fpath)
    res = process_single_month(df_month)
    month_results[label] = res
    n_surg = len(res['surgery_df'])
    n_adh = res['surgery_df']['is_adherent_1m'].sum()
    n_g_rep = res['surgery_df']['has_graft_resurgery_1m'].sum()
    n_all_rep = len(res['repeat_1m'])
    print(f'  [OK] {label}: {n_surg} Surgeries | 1M Adherent (11–45d): {n_adh} ({n_adh/n_surg*100:.1f}%) | Graft Resurgeries: {n_g_rep} ({n_g_rep/n_surg*100:.2f}%) | All Re-interventions: {n_all_rep}')

In [ ]:
# ── Build 6 Side-by-Side Trend Tables ──
trend_tables = build_3month_trend_tables(month_results)

print('✓ All 6 Trend Tables constructed successfully:')
for name in trend_tables:
    print(f'  • {name}')

In [ ]:
# ── Display Table 1: Adherence Trend Overall ──
print('=== 1. Adherence Trend Overall ===')
display(trend_tables['1_Trend_Adherence_Overall'])

In [ ]:
# ── Display Table 2: Adherence Trend by Campus ──
print('=== 2. Adherence Trend by Campus ===')
display(trend_tables['2_Trend_Adherence_Campus'])

In [ ]:
# ── Display Table 3: Adherence Trend by Surgery Procedure ──
print('=== 3. Adherence Trend by Surgery Procedure ===')
display(trend_tables['3_Trend_Adherence_SurgType'])

In [ ]:
# ── Display Table 4: Resurgery Trend Overall & Categories ──
print('=== 4. Resurgery Trend Overall & Categories ===')
display(trend_tables['4_Trend_Resurgery_Overall'])

In [ ]:
# ── Display Table 5 & 6: Resurgery Trend by Campus & Surgery ──
print('=== 5. Resurgery Trend by Campus ===')
display(trend_tables['5_Trend_Resurgery_Campus'])

print('=== 6. Resurgery Trend by Surgery Procedure ===')
display(trend_tables['6_Trend_Resurgery_SurgType'])

In [ ]:
# ── Generate & Export Executive Widescreen PowerPoint Presentation ──
pptx_buf = create_trend_pptx(trend_tables, month_labels)
with open(OUTPUT_PPTX, 'wb') as f:
    f.write(pptx_buf.getvalue())

print(f'[OK] Executive Widescreen PowerPoint Presentation saved to: {OUTPUT_PPTX}')